# Chapter 27: Agent Development Frameworks
## GAS in LangGraph vs CrewAI

> **Book:** Agentic AI Engineering  
> **Part 5:** Agentic Systems  
> **System:** Guardian Angel System (GAS) — framework comparison

---


## 🎯 1. Chapter Goal

By the end of this chapter you will:
- Understand the **5 major agent frameworks**: LangGraph, CrewAI, AutoGen/AG2, OpenAI Agents SDK, Pydantic AI
- Implement the **same GAS system in two frameworks**: LangGraph and CrewAI
- Compare them on **8 dimensions** with a side-by-side table
- Benchmark both on the **hypoglycemia scenario**: latency, token usage, output quality
- Build a **decision guide** for choosing frameworks in medical/safety-critical systems


## 📖 2. Theory Recap

### Framework Landscape

| Framework | Paradigm | State Model | Best For |
|-----------|----------|-------------|----------|
| **LangGraph** | Stateful directed graph | Explicit TypedDict | Complex workflows, auditability |
| **CrewAI** | Role-based crews | Implicit task context | Rapid prototyping, role clarity |
| **AutoGen/AG2** | Actor-based messaging | Message history | Research, multi-turn conversations |
| **OpenAI Agents SDK** | Simple agent loop | Thread + messages | Simple tasks, OpenAI-native |
| **Pydantic AI** | Type-safe agents | Pydantic models | Type safety, validation-heavy apps |

### Framework Selection Criteria

1. **State management**: Do you need explicit, auditable state? → LangGraph
2. **Team structure**: Do you think in roles? → CrewAI
3. **Conversation-first**: Multi-turn dialogue? → AutoGen
4. **Simplicity**: Minimal boilerplate? → OpenAI Agents SDK
5. **Type safety**: Strict validation? → Pydantic AI

### Framework Lock-in and Portability

**Lock-in risks:**
- LangGraph: tied to LangChain ecosystem
- CrewAI: proprietary task/crew abstractions
- AutoGen: Microsoft-backed, may diverge

**Portability strategy:** Keep business logic in pure Python functions; wrap with framework-specific nodes/agents.

### For Medical/Safety-Critical Systems

Key requirements:
- **Auditability**: Every decision must be traceable
- **Checkpointing**: Resume after failure without data loss
- **Explicit state**: No hidden state that can't be inspected
- **Determinism**: Same input → same output (or bounded randomness)
- **Human-in-the-loop**: Easy to add approval gates


In [ ]:
# Install dependencies
!pip install -q langgraph langchain-openai langchain-core crewai


In [ ]:
import os
import sys
import time
import operator
from datetime import datetime
from typing import TypedDict, Annotated, Literal

# ── TEST MODE ──────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))
from llm_utils import llm, TEST_MODE, BACKEND, MODEL


## 🔨 3. Build It

### Implementation 1: LangGraph GAS (Condensed)

We built the full LangGraph GAS in Chapter 26. Here's the condensed version showing the key design decisions.


In [ ]:
# ── LangGraph GAS — condensed implementation ──────────────────────────────────
from langgraph.graph import StateGraph, START, END as GRAPH_END
from langgraph.checkpoint.memory import MemorySaver


class GASState(TypedDict):
    patient_id: str
    glucose_value: float
    glucose_trend: str
    alert_level: str
    recommendation: str
    safety_approved: bool
    caregiver_notified: bool
    agent_messages: Annotated[list, operator.add]
    next_agent: str
    safety_score: float
    safety_issues: list
    debate_rounds: int


def _alert_level(glucose: float) -> str:
    if glucose < 54: return "critical"
    if glucose < 70: return "high"
    if glucose < 80: return "medium"
    if glucose <= 180: return "none"
    if glucose <= 250: return "low"
    return "high"


def _recommendation(glucose: float, alert: str, trend: str) -> str:
    if alert == "critical":
        return (f"IMMEDIATE ACTION: Glucose critically low at {glucose} mg/dL. "
                "Administer 15g fast-acting carbohydrates immediately. "
                "Call 911 if unresponsive. Recheck in 15 minutes.")
    if alert == "high":
        return (f"Glucose low at {glucose} mg/dL. Consume 15g carbohydrates. "
                "Avoid exercise. Recheck in 15 minutes.")
    if alert in ("medium", "low"):
        return f"Glucose at {glucose} mg/dL. Monitor closely. Recheck in 30 minutes."
    return f"Glucose normal at {glucose} mg/dL. Continue management. Check in 2 hours."


# ── Agent nodes (50-line graph definition) ────────────────────────────────────

def lg_supervisor(state): 
    ts = datetime.now().strftime("%H:%M:%S")
    return {"next_agent": "monitor",
            "agent_messages": [{"agent": "supervisor", "ts": ts, "msg": "routing to monitor"}]}

def lg_monitor(state):
    g = state["glucose_value"]
    alert = _alert_level(g)
    trend = "falling_fast" if g < 60 else "falling" if g < 80 else "stable"
    ts = datetime.now().strftime("%H:%M:%S")
    return {"alert_level": alert, "glucose_trend": trend,
            "next_agent": "advisor" if alert != "none" else "END",
            "agent_messages": [{"agent": "monitor", "ts": ts,
                                 "msg": f"glucose={g}, alert={alert}"}]}

def lg_advisor(state):
    g, alert, trend = state["glucose_value"], state["alert_level"], state["glucose_trend"]
    rec = _recommendation(g, alert, trend)
    ts = datetime.now().strftime("%H:%M:%S")
    return {"recommendation": rec, "next_agent": "safety",
            "agent_messages": [{"agent": "advisor", "ts": ts, "msg": "recommendation generated"}]}

def lg_safety(state):
    rec = state["recommendation"]
    issues = []
    if state["alert_level"] == "critical" and "immediate" not in rec.lower():
        issues.append("missing immediate action")
    if "recheck" not in rec.lower() and "check" not in rec.lower():
        issues.append("missing recheck")
    score = max(0.0, 1.0 - len(issues) * 0.2)
    approved = score >= 0.75
    debate = state.get("debate_rounds", 0)
    if not approved and debate < 2:
        next_a = "advisor"
    elif state["alert_level"] in ("high", "critical"):
        next_a = "caregiver"
    else:
        next_a = "END"
    ts = datetime.now().strftime("%H:%M:%S")
    return {"safety_approved": approved, "safety_score": score, "safety_issues": issues,
            "debate_rounds": debate + (1 if not approved else 0), "next_agent": next_a,
            "agent_messages": [{"agent": "safety", "ts": ts,
                                 "msg": f"score={score:.2f}, approved={approved}"}]}

def lg_caregiver(state):
    ts = datetime.now().strftime("%H:%M:%S")
    notif_id = f"NOTIF-{ts.replace(':', '')}"
    if TEST_MODE:
        print(f"  📱 [LangGraph] Caregiver notified: {notif_id}")
    return {"caregiver_notified": True, "next_agent": "END",
            "agent_messages": [{"agent": "caregiver", "ts": ts,
                                 "msg": f"notified: {notif_id}"}]}


def _route(state):
    n = state.get("next_agent", "END")
    if n == "advisor" and state.get("debate_rounds", 0) >= 2:
        return "caregiver" if state["alert_level"] in ("high", "critical") else "END"
    return n


# Build graph
b = StateGraph(GASState)
for name, fn in [("supervisor", lg_supervisor), ("monitor", lg_monitor),
                  ("advisor", lg_advisor), ("safety", lg_safety), ("caregiver", lg_caregiver)]:
    b.add_node(name, fn)

b.add_edge(START, "supervisor")
b.add_conditional_edges("supervisor", _route, {"monitor": "monitor", "END": GRAPH_END})
b.add_conditional_edges("monitor", _route, {"advisor": "advisor", "END": GRAPH_END})
b.add_conditional_edges("advisor", _route, {"safety": "safety", "END": GRAPH_END})
b.add_conditional_edges("safety", _route,
    {"advisor": "advisor", "caregiver": "caregiver", "END": GRAPH_END})
b.add_edge("caregiver", GRAPH_END)

lg_gas = b.compile(checkpointer=MemorySaver())

print("✅ LangGraph GAS compiled (~50 lines for graph definition)")
print("\nLangGraph strengths:")
print("  ✅ Explicit GASState TypedDict — full auditability")
print("  ✅ Conditional routing — different paths per alert level")
print("  ✅ MemorySaver checkpointing — resume after failure")
print("  ✅ Debate loop — Safety Critic sends Advisor back for revision")
print("  ✅ Visualization — gas_graph.get_graph().draw_mermaid()")


### Implementation 2: CrewAI GAS

CrewAI uses a **role-based** paradigm: you define agents by their role, goal, and backstory, then assign tasks.


In [ ]:
# ── CrewAI GAS implementation ─────────────────────────────────────────────────
# CrewAI uses role-based agents with tasks and a crew orchestrator

CREWAI_AVAILABLE = False
try:
    from crewai import Agent, Task, Crew, Process
    from langchain_openai import ChatOpenAI
    CREWAI_AVAILABLE = True
    print("✅ CrewAI available")
except ImportError:
    print("⚠️  CrewAI not installed. Showing code structure only.")
    print("   Install with: pip install crewai")


In [ ]:
# CrewAI GAS — code structure (runs if CrewAI is installed and API key present)

CREWAI_CODE = '''
from crewai import Agent, Task, Crew, Process
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ── Define 5 role-based agents ────────────────────────────────────────────────

monitor_agent = Agent(
    role="Glucose Monitor",
    goal="Continuously monitor patient glucose levels and detect anomalies",
    backstory=(
        "You are a specialized medical monitoring agent with expertise in "
        "continuous glucose monitoring (CGM) systems. You analyze glucose "
        "readings, compute trends, and classify alert levels according to "
        "ADA 2024 guidelines."
    ),
    llm=llm,
    verbose=True,
)

advisor_agent = Agent(
    role="Medical Advisor",
    goal="Provide evidence-based recommendations based on glucose readings",
    backstory=(
        "You are a diabetes management specialist with 20 years of clinical "
        "experience. You provide actionable, evidence-based recommendations "
        "grounded in ADA guidelines and the patient's medical history."
    ),
    llm=llm,
    verbose=True,
)

safety_agent = Agent(
    role="Safety Critic",
    goal="Validate all recommendations for patient safety before delivery",
    backstory=(
        "You are a medical safety officer specializing in AI-generated "
        "clinical recommendations. You check for contraindications, missing "
        "instructions, and unsafe advice."
    ),
    llm=llm,
    verbose=True,
)

caregiver_agent = Agent(
    role="Caregiver Notifier",
    goal="Escalate critical alerts to human caregivers via SMS and email",
    backstory=(
        "You are responsible for all human escalation in the Guardian Angel "
        "System. When glucose is critical, you format and send alerts to "
        "caregivers through multiple channels."
    ),
    llm=llm,
    verbose=True,
)

# ── Define tasks ──────────────────────────────────────────────────────────────

monitor_task = Task(
    description=(
        "Analyze the glucose reading of {glucose_value} mg/dL for patient {patient_id}. "
        "Determine the alert level (none/low/medium/high/critical) and glucose trend. "
        "Output: alert_level, trend, glucose_status"
    ),
    expected_output="Alert level, trend, and glucose status classification",
    agent=monitor_agent,
)

advise_task = Task(
    description=(
        "Based on the monitor's findings, generate a clinical recommendation "
        "for the patient. Follow ADA 2024 guidelines. Include: immediate action, "
        "carbohydrate amount if needed, recheck timing."
    ),
    expected_output="Actionable clinical recommendation (2-3 sentences)",
    agent=advisor_agent,
    context=[monitor_task],
)

validate_task = Task(
    description=(
        "Review the advisor's recommendation for safety. Check: "
        "(1) urgency appropriate for alert level, "
        "(2) no contraindicated advice, "
        "(3) recheck instruction present. "
        "Output: approved (yes/no), safety_score (0-1), issues list"
    ),
    expected_output="Safety validation result with score and any issues",
    agent=safety_agent,
    context=[advise_task],
)

notify_task = Task(
    description=(
        "If the alert level is high or critical, format and send a caregiver "
        "notification. Include: patient ID, glucose value, alert level, "
        "recommendation summary, timestamp. Simulate SMS and email delivery."
    ),
    expected_output="Notification confirmation with ID and delivery status",
    agent=caregiver_agent,
    context=[monitor_task, validate_task],
)

# ── Assemble the crew ─────────────────────────────────────────────────────────

gas_crew = Crew(
    agents=[monitor_agent, advisor_agent, safety_agent, caregiver_agent],
    tasks=[monitor_task, advise_task, validate_task, notify_task],
    process=Process.sequential,  # or Process.hierarchical with manager_llm
    verbose=True,
)

# Run the crew
result = gas_crew.kickoff(inputs={
    "patient_id": "PATIENT-001",
    "glucose_value": 42.0,
})
'''

print("CrewAI GAS code structure:")
print("  - 4 role-based agents (monitor, advisor, safety, caregiver)")
print("  - 4 tasks with context dependencies")
print("  - Sequential process (or hierarchical with manager LLM)")
print()

if CREWAI_AVAILABLE and BACKEND == "openai" and not TEST_MODE:
    print("Running CrewAI GAS...")
    exec(CREWAI_CODE)
else:
    print("[TEST MODE / No API key] Showing code structure only.")
    print("To run: install crewai + have Ollama running (or set OPENAI_API_KEY)")
    print()
    # Show the code
    print(CREWAI_CODE[:500] + "\n... (truncated for display)")


### Side-by-Side Comparison: LangGraph vs CrewAI


In [ ]:
# 8-dimension comparison table
comparison = [
    ("State Management",
     "Explicit TypedDict — every field typed and auditable",
     "Implicit task context — state lives in LLM conversation"),
    ("Routing",
     "Conditional edges — deterministic, code-controlled",
     "Sequential/hierarchical — LLM decides next step"),
    ("Checkpointing",
     "Built-in MemorySaver / PostgresSaver — resume after failure",
     "No built-in checkpointing — restart from scratch"),
    ("Auditability",
     "Full state trace at every node — perfect for compliance",
     "LLM conversation history — harder to audit"),
    ("Prototyping Speed",
     "Slower — requires explicit graph definition",
     "Faster — define role + goal + backstory in minutes"),
    ("Human-in-the-Loop",
     "interrupt_before/after — precise control",
     "Callback hooks — less granular"),
    ("Parallelism",
     "Parallel branches with Send() API",
     "Async tasks with Process.hierarchical"),
    ("Medical/Safety Use",
     "✅ Recommended — explicit state, checkpointing, determinism",
     "⚠️  Risky — implicit state, LLM routing, no checkpointing"),
]

print(f"{'Dimension':<22} {'LangGraph':<45} {'CrewAI':<45}")
print("─" * 112)
for dim, lg, crew in comparison:
    # Wrap long text
    print(f"{dim:<22} {lg[:44]:<45} {crew[:44]:<45}")
    if len(lg) > 44 or len(crew) > 44:
        lg2 = lg[44:] if len(lg) > 44 else ""
        crew2 = crew[44:] if len(crew) > 44 else ""
        if lg2 or crew2:
            print(f"{'':22} {lg2[:44]:<45} {crew2[:44]:<45}")
    print()


### Benchmark: Hypoglycemia Scenario


In [ ]:
# Benchmark: run LangGraph GAS on hypoglycemia scenario and measure performance

def make_lg_state(patient_id: str, glucose: float) -> GASState:
    return GASState(
        patient_id=patient_id,
        glucose_value=glucose,
        glucose_trend="stable",
        alert_level="none",
        recommendation="",
        safety_approved=False,
        caregiver_notified=False,
        agent_messages=[],
        next_agent="monitor",
        safety_score=0.0,
        safety_issues=[],
        debate_rounds=0,
    )


print("🏃 Benchmarking LangGraph GAS on hypoglycemia scenario (glucose=42 mg/dL)")
print("=" * 60)

# Run 3 times and average
lg_times = []
for run in range(3):
    state = make_lg_state("PATIENT-BENCH", 42.0)
    config = {"configurable": {"thread_id": f"bench-lg-{run}"}}
    t0 = time.time()
    final = lg_gas.invoke(state, config=config)
    elapsed = (time.time() - t0) * 1000
    lg_times.append(elapsed)

lg_avg = sum(lg_times) / len(lg_times)
print(f"\n📊 LangGraph Results:")
print(f"  Alert level:        {final['alert_level']}")
print(f"  Safety approved:    {final['safety_approved']}")
print(f"  Caregiver notified: {final['caregiver_notified']}")
print(f"  Agent messages:     {len(final['agent_messages'])}")
print(f"  Avg latency:        {lg_avg:.1f}ms (3 runs)")
print(f"  Recommendation:     {final['recommendation'][:80]}...")

# CrewAI benchmark (simulated in TEST_MODE)
print(f"\n📊 CrewAI Results (simulated — requires API key):")
print(f"  Alert level:        high (LLM-determined)")
print(f"  Safety approved:    True (LLM-validated)")
print(f"  Caregiver notified: True")
print(f"  Avg latency:        ~3500ms (3 LLM calls, no caching)")
print(f"  Token usage:        ~800 tokens (vs 0 for LangGraph in TEST_MODE)")

print(f"\n🏆 Comparison:")
print(f"  LangGraph latency:  {lg_avg:.0f}ms")
print(f"  CrewAI latency:     ~3500ms (estimated)")
print(f"  LangGraph tokens:   0 (rule-based in TEST_MODE)")
print(f"  CrewAI tokens:      ~800 (always uses LLM)")
print(f"  LangGraph state:    Explicit, typed, auditable")
print(f"  CrewAI state:       Implicit, in LLM context")


### Decision Guide: When to Use Each Framework


In [ ]:
decision_guide = {
    "Use LangGraph when": [
        "You need explicit, auditable state (medical, financial, legal)",
        "Workflow has complex conditional routing",
        "You need checkpointing and resume-after-failure",
        "Human-in-the-loop approval gates are required",
        "You need deterministic behavior for testing",
        "Compliance requires full audit trail",
    ],
    "Use CrewAI when": [
        "Rapid prototyping — get to demo in hours",
        "Team thinks in roles (researcher, writer, reviewer)",
        "Workflow is mostly sequential with clear task handoffs",
        "LLM reasoning should drive task sequencing",
        "No strict state management requirements",
    ],
    "Use AutoGen when": [
        "Multi-turn conversations between agents",
        "Research and exploration tasks",
        "Code generation with execution feedback loops",
    ],
    "Use OpenAI Agents SDK when": [
        "Simple single-agent tasks",
        "Minimal boilerplate needed",
        "Already using OpenAI ecosystem exclusively",
    ],
    "Use Pydantic AI when": [
        "Type safety is paramount",
        "Structured output validation is critical",
        "Python-first, minimal framework overhead",
    ],
}

for framework, criteria in decision_guide.items():
    print(f"\n🔧 {framework}:")
    for c in criteria:
        print(f"   ✓ {c}")

print("\n" + "=" * 60)
print("🏥 For GAS (medical safety-critical): LangGraph wins")
print("   Explicit state + checkpointing + audit trail = patient safety")


## 🏥 4. GAS Connection

**LangGraph wins for GAS because explicit state management and checkpointing are critical for patient safety.**

### Why LangGraph for Medical AI

| Requirement | LangGraph | CrewAI |
|-------------|-----------|--------|
| Audit trail | ✅ Full GASState at every node | ❌ LLM conversation history |
| Resume after failure | ✅ MemorySaver checkpointing | ❌ Restart from scratch |
| Deterministic routing | ✅ Code-controlled edges | ❌ LLM decides next step |
| Human approval gates | ✅ interrupt_before/after | ⚠️ Callback hooks |
| Compliance | ✅ Every state transition logged | ❌ Hard to audit |

### CrewAI's Role in the GAS Ecosystem

CrewAI is excellent for **prototyping** the GAS:
- Define the 5 agents in 50 lines
- Get a working demo in 2 hours
- Validate the agent roles and task flow

Then **migrate to LangGraph** for production:
- Add explicit state management
- Add checkpointing
- Add human approval gates
- Add compliance logging

**The GAS uses LangGraph in production, but CrewAI was used to prototype it.**


## ✅ 5. Checkpoint

### What We Built

| Component | Description |
|-----------|-------------|
| LangGraph GAS | Condensed 50-line graph definition |
| CrewAI GAS | Role-based implementation with 4 agents + 4 tasks |
| Comparison table | 8-dimension LangGraph vs CrewAI analysis |
| Benchmark | Latency and token usage comparison |
| Decision guide | When to use each of the 5 frameworks |

### Key Takeaways

- **LangGraph**: Explicit state, conditional routing, checkpointing → **production medical AI**
- **CrewAI**: Role-based, fast prototyping, LLM-driven → **rapid exploration**
- **Framework lock-in**: Keep business logic in pure Python; wrap with framework nodes
- **For GAS**: LangGraph wins on auditability, checkpointing, and determinism

### Next Chapter

**Chapter 28: Agentic UI** — We'll build a Streamlit dashboard for the GAS system with live glucose charts, agent decision feeds, and human approval gates.
